# Fixture ID Mapping (Sportradar ↔ OddsJam)

Build a 1:1 crosswalk between Sportradar `sport_event_id` and OddsJam `fixture_id`.

**Matching rules**
- Collapse consensus to match-level player pairs (moneyline first; fill from other player markets if needed)
- Normalize names to sorted underscore tokens on both sides
- Require both players to match as a set
- Keep candidates within ±12 hours (`first_event_time` vs `start_date`)
- Accept only mutual rank-1 matches with a clear time margin

Matching helpers live in `refined_tables/fixture_id_mapping/build.py` for later backfill extraction.

In [2]:
from injestion.core.env import load_env
load_env()

import os
os.environ["GOOGLE_CLOUD_PROJECT"] = "prizepicksanalytics"
# If needed, also set your service-account key path:
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/absolute/path/to/key.json"

from google.cloud import bigquery
bq_client = bigquery.Client(project=os.environ["GOOGLE_CLOUD_PROJECT"])

In [3]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "injestion").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError(f"Could not find project root from {cwd}")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from injestion.core.bq import get_client
from injestion.core.env import load_env

load_env()
bq_client = get_client()
print(f"Project root: {project_root}")
print(f"BQ project: {bq_client.project}")

Project root: /Users/matt.holden/Projects/tennis-origination
BQ project: prizepicksanalytics


In [4]:
from refined_tables import get_table_id

FIXTURE_STATS_TABLE_ID = get_table_id("fixture_stats")
CONSENSUS_TABLE_ID = get_table_id("consensus")
MAPPING_TABLE_ID = get_table_id("fixture_id_mapping")

print(f"fixture_stats: {FIXTURE_STATS_TABLE_ID}")
print(f"consensus:     {CONSENSUS_TABLE_ID}")
print(f"mapping:       {MAPPING_TABLE_ID}")

fixture_stats: prizepicksanalytics.originations_tennis.refined_fixture_stats
consensus:     prizepicksanalytics.originations_tennis.refined_consensus
mapping:       prizepicksanalytics.originations_tennis.refined_fixture_id_mapping


## Load source tables

In [5]:
fixture_stats_df = bq_client.query(
    f"""
    SELECT
      sport_event_id,
      home_competitor_name,
      away_competitor_name,
      first_event_time
    FROM `{FIXTURE_STATS_TABLE_ID}`
    """
).to_dataframe()

print(f"fixture_stats rows: {len(fixture_stats_df):,}")
fixture_stats_df.head()

fixture_stats rows: 9,480


,sport_event_id,home_competitor_name,away_competitor_name,first_event_time
0,sr:sport_event:58213339,"Navarro, Emma","Arango, Emiliana",2025-03-02 23:11:15+00:00
1,sr:sport_event:58906357,"Gauff, Coco","Kenin, Sofia",2025-03-20 17:51:32+00:00
2,sr:sport_event:59414837,"Volynets, Katie","Sebov, Katherine",2025-03-31 16:44:32+00:00
3,sr:sport_event:59551117,"Dimitrov, Grigor","de Minaur, Alex",2025-04-11 13:25:55+00:00
4,sr:sport_event:60130357,"Shnaider, Diana","Sevastova, Anastasija",2025-04-26 09:07:06+00:00


In [6]:
consensus_df = bq_client.query(
    f"""
    SELECT
      fixture_id,
      start_date,
      market,
      name,
      normalized_selection,
      normalized_selection_key
    FROM `{CONSENSUS_TABLE_ID}`
    """
).to_dataframe()

print(f"consensus rows: {len(consensus_df):,}")
consensus_df.head()

consensus rows: 582,979


,fixture_id,start_date,market,name,normalized_selection,normalized_selection_key
0,20260624B537891E,2026-06-24 15:30:00+00:00,Moneyline,Abdullah Shelbayh,abdullah_shelbayh,abdullah_shelbayh
1,20260621CF6025AC,2026-06-21 12:30:00+00:00,Moneyline,Abdullah Shelbayh,abdullah_shelbayh,abdullah_shelbayh
2,2026062078FC4AF5,2026-06-20 10:30:00+00:00,1st Set Game 2 Moneyline,Abdullah Shelbayh,abdullah_shelbayh,abdullah_shelbayh
3,2026062078FC4AF5,2026-06-20 10:30:00+00:00,Moneyline,Abdullah Shelbayh,abdullah_shelbayh,abdullah_shelbayh
4,2026062078FC4AF5,2026-06-20 10:30:00+00:00,1st Set Game 1 Moneyline,Abdullah Shelbayh,abdullah_shelbayh,abdullah_shelbayh


## Build match-level frames + join

In [7]:
from refined_tables.fixture_id_mapping import (
    assert_one_to_one,
    build_oddsjam_fixture_pairs,
    format_accepted_for_upload,
    match_fixtures,
    normalize_player_name,
    prepare_sportradar_matches,
)

# Sanity-check name normalization
for raw in ["Wong, Hong Yi Cody", "Abdullah Shelbayh", "abdullah_shelbayh"]:
    print(f"{raw!r:40s} -> {normalize_player_name(raw)}")

'Wong, Hong Yi Cody'                     -> cody_hong_wong_yi
'Abdullah Shelbayh'                      -> abdullah_shelbayh
'abdullah_shelbayh'                      -> abdullah_shelbayh


In [8]:
sr_matches = prepare_sportradar_matches(fixture_stats_df)
oj_pairs, oj_insufficient = build_oddsjam_fixture_pairs(consensus_df)

print(f"Sportradar match rows:     {len(sr_matches):,}")
print(f"OddsJam pair rows:         {len(oj_pairs):,}")
print(f"OddsJam insufficient:      {len(oj_insufficient):,}")
if not oj_insufficient.empty:
    display(oj_insufficient.head(10))

display(sr_matches.head())
display(oj_pairs.head())

Sportradar match rows:     9,480
OddsJam pair rows:         15,953
OddsJam insufficient:      11


,fixture_id,start_date,n_moneyline_players,n_all_players,reason
0,20260302C16FE83E,2026-03-02 23:05:00+00:00,3,3,insufficient_players
1,202509256915C118,2025-09-25 05:00:00+00:00,3,3,insufficient_players
2,2026022424AC8888,2026-02-24 22:30:00+00:00,3,3,insufficient_players
3,202510045E187D2B,2025-10-04 08:00:00+00:00,1,1,insufficient_players
4,wta:76E089325035,2025-01-20 08:15:00+00:00,4,4,insufficient_players
5,0A38AA39CDA9,2024-12-04 10:55:00+00:00,1,1,insufficient_players
6,202510197D1E1959,2025-10-19 18:55:00+00:00,1,1,insufficient_players
7,20251025A715BA87,2025-10-25 08:00:00+00:00,1,1,insufficient_players
8,20251013807D8600,2025-10-13 11:30:00+00:00,0,1,insufficient_players
9,202506259958E070,2025-06-25 12:15:00+00:00,0,1,insufficient_players


,sport_event_id,away_competitor_name,first_event_time,home_competitor_name,norm_home,norm_away,pair_key
0,sr:sport_event:58213339,"Arango, Emiliana",2025-03-02 23:11:15+00:00,"Navarro, Emma",emma_navarro,arango_emiliana,arango_emiliana|emma_navarro
1,sr:sport_event:58906357,"Kenin, Sofia",2025-03-20 17:51:32+00:00,"Gauff, Coco",coco_gauff,kenin_sofia,coco_gauff|kenin_sofia
2,sr:sport_event:59414837,"Sebov, Katherine",2025-03-31 16:44:32+00:00,"Volynets, Katie",katie_volynets,katherine_sebov,katherine_sebov|katie_volynets
3,sr:sport_event:59551117,"de Minaur, Alex",2025-04-11 13:25:55+00:00,"Dimitrov, Grigor",dimitrov_grigor,alex_de_minaur,alex_de_minaur|dimitrov_grigor
4,sr:sport_event:60130357,"Sevastova, Anastasija",2025-04-26 09:07:06+00:00,"Shnaider, Diana",diana_shnaider,anastasija_sevastova,anastasija_sevastova|diana_shnaider


,fixture_id,start_date,oj_player_a,oj_player_b,pair_key,player_source
0,20260624B537891E,2026-06-24 15:30:00+00:00,abdullah_shelbayh,dimitrov_grigor,abdullah_shelbayh|dimitrov_grigor,moneyline
1,20260621CF6025AC,2026-06-21 12:30:00+00:00,abdullah_shelbayh,marc_polmans,abdullah_shelbayh|marc_polmans,moneyline
2,2026062078FC4AF5,2026-06-20 10:30:00+00:00,abdullah_shelbayh,alexander_shevchenko,abdullah_shelbayh|alexander_shevchenko,moneyline
3,atp:D56EF721FD32,2025-02-18 11:30:00+00:00,abdullah_shelbayh,botic_de_van_zandschulp,abdullah_shelbayh|botic_de_van_zandschulp,moneyline
4,2025090614D95DE0,2025-09-06 16:00:00+00:00,abril_cardenas_olivares,elena_pridankina,abril_cardenas_olivares|elena_pridankina,moneyline


In [9]:
TIME_WINDOW_HOURS = 12.0
TIME_MARGIN_MINUTES = 120.0

results = match_fixtures(
    sr_matches,
    oj_pairs,
    time_window_hours=TIME_WINDOW_HOURS,
    time_margin_minutes=TIME_MARGIN_MINUTES,
)

accepted_raw = results["accepted"]
ambiguous = results["ambiguous"]
unmatched_sr = results["unmatched_sr"]
candidates = results["candidates"]

print(f"Candidates in window: {len(candidates):,}")
print(f"Accepted:             {len(accepted_raw):,}")
print(f"Ambiguous:            {len(ambiguous):,}")
print(f"Unmatched SR:         {len(unmatched_sr):,}")

accepted_df = format_accepted_for_upload(accepted_raw)
assert_one_to_one(accepted_df)
print("1:1 uniqueness check passed")
accepted_df.head(20)

Candidates in window: 7,675
Accepted:             7,673
Ambiguous:            2
Unmatched SR:         1,807
1:1 uniqueness check passed


,sport_event_id,fixture_id,first_event_time,start_date,time_delta_minutes,home_competitor_name,away_competitor_name,norm_home,norm_away,oj_player_a,oj_player_b,match_method,updated_at
0,sr:sport_event:58213339,20250302C21A609F,2025-03-02 23:11:15+00:00,2025-03-02 23:00:00+00:00,11.250000,"Navarro, Emma","Arango, Emiliana",emma_navarro,arango_emiliana,arango_emiliana,emma_navarro,both_players_token_set,2026-09-25 02:15:52.217909+00:00
1,sr:sport_event:58906357,2025032184012346,2025-03-20 17:51:32+00:00,2025-03-20 17:40:00+00:00,11.533333,"Gauff, Coco","Kenin, Sofia",coco_gauff,kenin_sofia,coco_gauff,kenin_sofia,both_players_token_set,2026-09-25 02:15:52.217909+00:00
2,sr:sport_event:59414837,202503315F7479E0,2025-03-31 16:44:32+00:00,2025-03-31 16:35:00+00:00,9.533333,"Volynets, Katie","Sebov, Katherine",katie_volynets,katherine_sebov,katherine_sebov,katie_volynets,both_players_token_set,2026-09-25 02:15:52.217909+00:00
3,sr:sport_event:59551117,2025041192E273D4,2025-04-11 13:25:55+00:00,2025-04-11 13:15:00+00:00,10.916667,"Dimitrov, Grigor","de Minaur, Alex",dimitrov_grigor,alex_de_minaur,alex_de_minaur,dimitrov_grigor,both_players_token_set,2026-09-25 02:15:52.217909+00:00
4,sr:sport_event:60130357,202504266A20CEDB,2025-04-26 09:07:06+00:00,2025-04-26 09:00:00+00:00,7.100000,"Shnaider, Diana","Sevastova, Anastasija",diana_shnaider,anastasija_sevastova,anastasija_sevastova,diana_shnaider,both_players_token_set,2026-09-25 02:15:52.217909+00:00
5,sr:sport_event:60276159,20250508AA660B31,2025-05-08 10:37:57+00:00,2025-05-08 10:30:00+00:00,7.950000,"Shnaider, Diana","Dolehide, Caroline",diana_shnaider,caroline_dolehide,caroline_dolehide,diana_shnaider,both_players_token_set,2026-09-25 02:15:52.217909+00:00
6,sr:sport_event:60645979,202505197B64B5A9,2025-05-19 08:11:07+00:00,2025-05-19 08:00:00+00:00,11.116667,"Yao, Xinxin","Andreescu, Bianca",xinxin_yao,andreescu_bianca,andreescu_bianca,xinxin_yao,both_players_token_set,2026-09-25 02:15:52.217909+00:00
7,sr:sport_event:60646037,2025052018EDCBE8,2025-05-20 13:19:06+00:00,2025-05-20 13:10:00+00:00,9.100000,"Sharma, Astra","Bektas, Emina",astra_sharma,bektas_emina,astra_sharma,bektas_emina,both_players_token_set,2026-09-25 02:15:52.217909+00:00
8,sr:sport_event:60736167,20250525AF1E7DC0,2025-05-27 09:10:11+00:00,2025-05-27 09:00:00+00:00,10.183333,"Wickmayer, Yanina","Azarenka, Victoria",wickmayer_yanina,azarenka_victoria,azarenka_victoria,wickmayer_yanina,both_players_token_set,2026-09-25 02:15:52.217909+00:00
9,sr:sport_event:61428049,2025062448E406B0,2025-06-24 14:08:41+00:00,2025-06-24 14:00:00+00:00,8.683333,"Sasnovich, Aliaksandra","Martinez Cirez, Carlota",aliaksandra_sasnovich,carlota_cirez_martinez,aliaksandra_sasnovich,carlota_cirez_martinez,both_players_token_set,2026-09-25 02:15:52.217909+00:00


## QC

In [10]:
n_sr = len(sr_matches)
n_oj = len(oj_pairs)
n_acc = len(accepted_df)

print(f"SR coverage: {n_acc / n_sr:.1%} ({n_acc:,} / {n_sr:,})" if n_sr else "SR coverage: n/a")
print(f"OJ coverage: {n_acc / n_oj:.1%} ({n_acc:,} / {n_oj:,})" if n_oj else "OJ coverage: n/a")
print(f"Insufficient OJ fixtures: {len(oj_insufficient):,}")

if not accepted_df.empty:
    deltas = accepted_df["time_delta_minutes"].abs()
    print("\n|time_delta_minutes| summary (accepted):")
    print(deltas.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_string())

if not ambiguous.empty:
    print(f"\nAmbiguous sample ({len(ambiguous):,} rows):")
    display(
        ambiguous[
            [
                c
                for c in [
                    "sport_event_id",
                    "fixture_id",
                    "home_competitor_name",
                    "away_competitor_name",
                    "abs_time_delta_minutes",
                    "rank_by_sr",
                    "rank_by_oj",
                ]
                if c in ambiguous.columns
            ]
        ].head(20)
    )

SR coverage: 80.9% (7,673 / 9,480)
OJ coverage: 48.1% (7,673 / 15,953)
Insufficient OJ fixtures: 11

|time_delta_minutes| summary (accepted):
count    7673.000000
mean        7.793751
std        14.204742
min         0.000000
50%         6.600000
90%        11.683333
95%        13.923333
99%        28.544667
max       506.116667

Ambiguous sample (2 rows):


,sport_event_id,fixture_id,home_competitor_name,away_competitor_name,abs_time_delta_minutes,rank_by_sr,rank_by_oj
0,sr:sport_event:64619540,2025101978BA3734,"Fritz, Taylor","Vacherot, Valentin",4.916667,1,1
1,sr:sport_event:64619540,2025102278BA3734,"Fritz, Taylor","Vacherot, Valentin",4.916667,1,1


In [11]:
# Spot-check random accepted links
if not accepted_df.empty:
    sample_n = min(20, len(accepted_df))
    display(
        accepted_df.sample(sample_n, random_state=42)[
            [
                "sport_event_id",
                "fixture_id",
                "home_competitor_name",
                "away_competitor_name",
                "oj_player_a",
                "oj_player_b",
                "first_event_time",
                "start_date",
                "time_delta_minutes",
            ]
        ].sort_values("time_delta_minutes")
    )
else:
    print("No accepted rows to spot-check.")

,sport_event_id,fixture_id,home_competitor_name,away_competitor_name,oj_player_a,oj_player_b,first_event_time,start_date,time_delta_minutes
5865,sr:sport_event:71486080,202605239DFFB10D,"Navone, Mariano","Tien, Learner",learner_tien,mariano_navone,2026-05-23 13:04:55+00:00,2026-05-23 13:10:00+00:00,-5.083333
1374,sr:sport_event:70669676,20260417E03BE7D9,"Norrie, Cameron","Jodar, Rafael",cameron_norrie,jodar_rafael,2026-04-17 15:46:45+00:00,2026-04-17 15:50:00+00:00,-3.250000
1176,sr:sport_event:63131415,20250824FB823EB9,"Anisimova, Amanda","Birrell, Kimberly",amanda_anisimova,birrell_kimberly,2025-08-26 17:58:42+00:00,2025-08-26 18:00:00+00:00,-1.300000
7152,sr:sport_event:71563728,2026060135DC71DA,"Auger-Aliassime, Felix","Tabilo, Alejandro",alejandro_tabilo,aliassime_auger_felix,2026-06-01 15:11:20+00:00,2026-06-01 15:10:00+00:00,1.333333
3955,sr:sport_event:63017911,202508189D7D021B,"Dmitruk, Kristina","Juvan, Kaja",dmitruk_kristina,juvan_kaja,2025-08-18 19:57:34+00:00,2025-08-18 19:55:00+00:00,2.566667
2753,sr:sport_event:62612404,20250805CB52E3FA,"Kukushkin, Mikhail","Nava, Emilio",emilio_nava,kukushkin_mikhail,2025-08-06 15:33:14+00:00,2025-08-06 15:30:00+00:00,3.233333
6300,sr:sport_event:60023469,20250421C2CC2090,"De Jong, Jesper","Boyer, Tristan",boyer_tristan,de_jesper_jong,2025-04-21 08:03:41+00:00,2025-04-21 08:00:00+00:00,3.683333
1731,sr:sport_event:61428059,202506247299B9AC,"Lazaro Garcia, Andrea","Gibson, Talia",andrea_garcia_lazaro,gibson_talia,2025-06-24 10:04:16+00:00,2025-06-24 10:00:00+00:00,4.266667
4615,sr:sport_event:72169280,20260625BA81452D,"Muchova, Karolina","Tauson, Clara",clara_tauson,karolina_muchova,2026-06-25 15:14:33+00:00,2026-06-25 15:10:00+00:00,4.550000
2111,sr:sport_event:64434401,2025101232596CC0,"Giron, Marcos","Bellucci, Mattia",bellucci_mattia,giron_marcos,2025-10-13 13:34:44+00:00,2025-10-13 13:30:00+00:00,4.733333


## Upload accepted crosswalk

Requires `BIGQUERY_REFINED_FIXTURE_ID_MAPPING_TABLE_ID` in `.env`.
Uses `WRITE_TRUNCATE` (full rebuild).

In [12]:
from refined_tables import replace_table
from refined_tables.schema import fixture_id_mapping as schema_mapping

assert_one_to_one(accepted_df)

if accepted_df.empty:
    raise ValueError("Refusing to upload an empty accepted crosswalk.")

job = replace_table(
    bq_client,
    MAPPING_TABLE_ID,
    accepted_df,
    schema=schema_mapping.get_schema(),
)
print(f"Uploaded {len(accepted_df):,} rows to {MAPPING_TABLE_ID}")
job

/Users/matt.holden/Projects/tennis-origination/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Uploaded 7,673 rows to prizepicksanalytics.originations_tennis.refined_fixture_id_mapping


LoadJob<project=prizepicksanalytics, location=US, id=fd1eb499-55f7-48ba-8f5a-77d8228187bd>